# View answercarefully_simple.sqlite3 (Colab)

Downloads `My Drive/ez-penta/answercarefully_simple.sqlite3` (made by `jb_simple_no_llm.ipynb`) and shows the table and its new word-split and flake columns.

No GPU needed: any Colab server works (**Select Kernel → Colab → New Colab Server**, CPU is fine).

In [ ]:
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/ez-penta")
DB = DRIVE_DIR / "answercarefully_simple.sqlite3"
WORK_DIR = Path("/content/work")

## 1. Download the db from Google Drive

In [ ]:
import shutil

if not DRIVE_DIR.exists():
    from google.colab import drive
    drive.mount("/content/drive")
if not DB.exists():
    raise FileNotFoundError(f"{DB} not found. Run jb_simple_no_llm.ipynb first.")

# SQLite is unreliable directly on the Drive mount, so read a local copy.
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DB = WORK_DIR / DB.name
shutil.copy(DB, LOCAL_DB)
print(f"Copied {DB} -> {LOCAL_DB}")

In [ ]:
import json
import sqlite3

import pandas as pd

connection = sqlite3.connect(LOCAL_DB)
records = pd.read_sql_query("SELECT * FROM records", connection)
generation_info = pd.read_sql_query("SELECT * FROM generation_info", connection)
connection.close()

NEW_COLUMNS = generation_info["columns"].iloc[0].split(", ")
FLAKE_COLUMNS = [column for column in NEW_COLUMNS if column.startswith("flake_")]
print(f"{len(records)} records, {len(records.columns)} columns, {len(NEW_COLUMNS)} new")
generation_info

## 2. The table

In [ ]:
# Column order, with the new columns marked.
pd.DataFrame({"column": records.columns, "new": records.columns.isin(NEW_COLUMNS)})

In [ ]:
pd.set_option("display.max_colwidth", 80)
records.head(10)

## 3. The new columns

In [ ]:
# One prompt with every new column, read top to bottom.
pd.set_option("display.max_colwidth", None)
row = 1
records.loc[row, ["text", "text_en", *NEW_COLUMNS]].to_frame(f"row {row}")

In [ ]:
# Word splits: stored as JSON lists, so decode them to count words.
text_words = records["text_words"].map(json.loads)
text_en_words = records["text_en_words"].map(json.loads)
display(pd.DataFrame({"japanese words": text_words.str.len(), "english words": text_en_words.str.len()}).describe().round(1))
records.assign(text_words=text_words, text_en_words=text_en_words)[["text", "text_words", "text_en", "text_en_words"]].head(5)

In [ ]:
# Japanese flakes next to the Japanese prompt.
records[["text", *[column for column in FLAKE_COLUMNS if "_ja_" in column]]].head(5)

In [ ]:
# English flakes next to the English prompt.
records[["text_en", *[column for column in FLAKE_COLUMNS if "_en_" in column]]].head(5)

In [ ]:
# How many flakes each column added per prompt (every10 grows with length; quartile is always 4).
def flake_count(column):
    flake = "はい" if column.endswith("_hai") else "yes"
    source = records["text" if "_ja_" in column else "text_en"]
    return (records[column].str.len() - source.str.len()) // len(flake)


pd.DataFrame({column: flake_count(column) for column in FLAKE_COLUMNS}).describe().round(1)